# FCC Smoke Test

This notebook performs a minimal black-box validation check against the public Semulo API endpoint. It verifies service health, CPU-only runtime metadata, an FCC torus summary result, and independent CSV recomputation of the reported time-window observable.

Implementation details and update rules are not included in this notebook.

In [ ]:
import io

import pandas as pd
import requests

BASE = "https://api.semulo.com"

payload = {
    "nx": 8,
    "ny": 8,
    "nz": 8,
    "seed": 0,
    "material_condition": "z_gradient",
}

BASE, payload

## Service And Runtime

The `/system` endpoint is part of the validation record. It should report an x86 Linux runtime and `gpu_available: False`.

In [ ]:
health = requests.get(f"{BASE}/health", timeout=30).json()
system = requests.get(f"{BASE}/system", timeout=30).json()

print(health)
print(system)

assert health["ok"] is True
assert system["ok"] is True
assert system["gpu_available"] is False

## FCC Summary

The JSON endpoint returns the headline time-window nearest-neighbor antiphase observable and timing.

In [ ]:
summary_response = requests.post(f"{BASE}/observe/fcc/torus", json=payload, timeout=120)
summary_response.raise_for_status()
summary = summary_response.json()

print("version:", summary["version"])
print("sites:", summary["sites"])
print("bonds:", summary["bonds"])
print("observable:", summary["observable"])
print("timing:", summary["timing"])

assert summary["ok"] is True
assert summary["sites"] == 256
assert summary["bonds"] == 1536
assert summary["cpu_only"] is True

## CSV Evidence

The final-state CSVs expose site and bond tables. The windowed bond CSV exposes per-bond counts over the same sample window used by the JSON summary.

In [ ]:
sites_response = requests.post(f"{BASE}/observe/fcc/torus/sites.csv", json=payload, timeout=120)
bonds_response = requests.post(f"{BASE}/observe/fcc/torus/bonds.csv", json=payload, timeout=120)
window_response = requests.post(f"{BASE}/observe/fcc/torus/bonds_window.csv", json=payload, timeout=120)

sites_response.raise_for_status()
bonds_response.raise_for_status()
window_response.raise_for_status()

sites = pd.read_csv(io.StringIO(sites_response.text))
bonds = pd.read_csv(io.StringIO(bonds_response.text))
bonds_window = pd.read_csv(io.StringIO(window_response.text))

print(sites.head())
print(bonds.head())
print(bonds_window.head())
print(len(sites), len(bonds), len(bonds_window))

assert len(sites) == summary["sites"]
assert len(bonds) == summary["bonds"]
assert len(bonds_window) == summary["bonds"]

## Independent Recompute

The mean of `window_antiphase_fraction` in the CSV should exactly match the JSON `antiphase_fraction` for the same payload.

In [ ]:
json_fraction = summary["observable"]["antiphase_fraction"]
csv_fraction = bonds_window["window_antiphase_fraction"].mean()
difference = abs(json_fraction - csv_fraction)
final_fraction = bonds["final_is_antiphase"].mean()

print("JSON window fraction:", json_fraction)
print("CSV window fraction:", csv_fraction)
print("difference:", difference)
print("final-state fraction:", final_fraction)

assert difference < 1e-12